# Spyre Cost Model — Elementwise Kernels

A **high-level, relative** performance model that predicts the **device latency** of a Spyre
kernel directly from its **LoopLevel IR** (the compiler's mid-level representation, *after*
the pre-scheduling passes). It is deliberately **not a simulator** — the goal is to predict
*which compiler choice is faster*, with a handful of physically-meaningful parameters, to
guide higher-level optimization.

This notebook (1) states the model, then (2) shows the experiments — run on real Spyre
hardware — that calibrate and validate it for **pointwise** ops.

> Self-contained: needs only `numpy` + `matplotlib`. All numbers are real device measurements.

## The model

For one kernel:

`T  ≈  T_fixed (~20 µs)  +  HBM_bytes / BW_HBM`   (LX traffic ≈ free)

- **`T_fixed` (~20 µs)** — a fixed per-kernel cost (device setup/pipeline fill + ~7 µs host
  dispatch). **Op-independent.**
- **Memory traffic** — every tensor argument (each *input read* + the *output write*) is one
  'pass'; bytes are charged to **HBM** or **LX (on-chip scratchpad)** by where the compiler
  placed them. LX-resident tensors don't touch HBM. **Broadcast and scalar inputs are cached**
  on-chip (loaded once) → they add ~no HBM traffic and are excluded (see (6)).
- **`BW_HBM ≈ 111 GB/s`** — the effective **balanced read+write** rate (the common case for
  pointwise). It is *not* the DRAM peak: read-only streams at ~176 and write-only at ~146, but
  mixing reads and writes ~halves it (see (5)). **LX traffic is ~free** (per-op cost below the
  measurement noise; LX is ~29× HBM).

**Why so simple?** Pointwise ops are **memory-bandwidth bound** — the arithmetic hides under
the streaming, so latency is essentially *bytes moved ÷ bandwidth* plus a fixed term.

**Measurement:** Spyre is a static-dataflow engine ⇒ device latency is **deterministic**. We
take the **min over 100 runs** of the per-kernel device time (host jitter only adds time).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- the model (calibrated from hardware) -------------------------------
T_FIXED_US = 20.0
BW_HBM = 111.0   # GB/s, effective BALANCED read+write rate (== bytes/ns)
DT = 2           # fp16 bytes

def predict_us(out_elems, n_streams, lx_passes=0):
    # n_streams = HBM memory passes (input reads + output write); broadcast/scalar
    # inputs are cached (excluded). LX-resident traffic is ~free.
    hbm = n_streams * out_elems * DT
    return T_FIXED_US + hbm / BW_HBM / 1000.0

# ---- experiment data (real Spyre device min, us; fp16) ------------------
ROWS = 512
gelu_sweep = [(ROWS*512,27.467),(ROWS*1024,37.878),(ROWS*2048,59.217),(ROWS*4096,92.851),(ROWS*8192,169.209)]
mul_sweep  = [(ROWS*512,38.239),(ROWS*1024,56.908),(ROWS*2048,93.200),(ROWS*4096,162.398),(ROWS*8192,317.637)]
arith = {'relu':39.655,'gelu':39.211,'sigmoid':38.730,'exp':38.355}
cores_sweep = [(1,61.001),(2,40.644),(4,43.986),(8,44.206),(16,40.283),(32,37.784)]
lx_chain = [(1,0,33.397),(2,2,33.500),(4,6,36.369),(8,14,36.281),(16,30,46.763)]
accuracy = [('gelu',ROWS*1024,2,38.427),('relu',ROWS*1024,2,39.655),('exp',ROWS*1024,2,38.355),
            ('sigmoid',ROWS*1024,2,38.730),('mul',ROWS*1024,3,57.258),('add',ROWS*1024,3,57.816)]
softmax_lx = {'LX on':71.05,'LX off':91.23}

# ---- (5) DRAM bandwidth: read-only vs write-only vs balanced (Rung 8) ----
PEAK_GBPS = 204.8   # LPDDR5 aggregate peak (_HBM_BW_GBS in work_division.py)
# effective BW asymptote (GB/s) at large size (slope of the two largest sizes):
bw_modes = {'read-only\n(sum)': 176.0, 'write-only\n(b[1,N]+c[N,1])': 146.0,
            'read+write\n(neg, x+1)': 97.0}

# ---- (6) broadcast I/O: full 2nd operand vs [1,N] broadcast (Rung 6) -----
# device min (us) at 512x1024: broadcasting the 2nd operand drops a full HBM pass.
bcast_pairs = {'add\n(a+b)': 58.0, 'bcast\n(a+b[1,N])': 36.0,
               'mul\n(a*b)': 58.0, 'mulbcast\n(a*b[1,N])': 35.0}
TWO_PASS_US = predict_us(ROWS*1024, 2)   # 1 read + 1 write (broadcast cached)

print('gelu[512x1024] predicted =', round(predict_us(512*1024, 2),1), 'us  (measured 37.9)')


## (1) Latency is linear in HBM bytes — memory-bandwidth bound
With LX planning off (all tensors in HBM), device latency is a straight line in HBM bytes:
`T = fixed + bytes / BW`. The slope is `1/BW`. Two stream patterns give two slopes — the
3-stream op (`mul`) has a *lower* effective bandwidth than the 2-stream op (`gelu`).

In [ ]:
fig, ax = plt.subplots(figsize=(7,4.5))
for sweep, ns, name, c in [(gelu_sweep,2,'gelu (1R+1W, 2 passes)','C0'),
                            (mul_sweep,3,'mul (2R+1W, 3 passes)','C3')]:
    mb = np.array([ns*e*DT for e,_ in sweep])/1e6
    meas = np.array([t for _,t in sweep])
    ax.plot(mb, meas, 'o', color=c, label=name+' (measured)')
# one BALANCED-BW model line: gelu sits on it; mul runs ~16% above -> the open anomaly
xs = np.linspace(0, 27, 50)
ax.plot(xs, T_FIXED_US + xs*1e6/BW_HBM/1000.0, 'k-', alpha=.6,
        label=f'model: 20us + bytes/{BW_HBM:.0f} GB/s')
ax.set_xlabel('HBM traffic moved (MB)'); ax.set_ylabel('device latency (us)')
ax.set_title('Pointwise latency is linear in HBM bytes (bandwidth-bound)')
ax.legend(); ax.grid(alpha=.3); plt.tight_layout(); plt.show()


## (2) Arithmetic is essentially free
Four different activations at the same shape (512×1024) take the **same** time — the op's
math is hidden under the memory streaming. This is why a single memory model works.

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
ops = list(arith); vals = [arith[o] for o in ops]
ax.bar(ops, vals, color='C0')
ax.axhline(np.mean(vals), ls='--', color='k', alpha=.6, label=f'mean {np.mean(vals):.1f} us')
for i,v in enumerate(vals): ax.text(i, v+0.6, f'{v:.1f}', ha='center')
ax.set_ylabel('device latency (us)'); ax.set_ylim(0, max(vals)*1.2)
ax.set_title('Arithmetic is free: latency independent of the activation\n(gelu[512x1024]-class, memory-bound)')
ax.legend(); plt.tight_layout(); plt.show()

## (3) HBM bandwidth is *shared* (flat past 2 cores); LX is ~free
**Left:** sweeping the number of cores for one memory-bound op — latency drops 1→2 cores,
then **flattens**: HBM bandwidth is shared and saturates by ~2 cores, so more cores don't
help. **Right:** chaining N gelus with every intermediate kept in LX — 16 chained ops ≈ 1,
because LX traffic is nearly free (~29× HBM bandwidth).

**Guess (open, unverified):** the effective HBM bandwidth here (~111 GB/s) is only about half the >200 GB/s DRAM peak. Because just ~2 cores already saturate it (the flat curve), the limiter looks like a **shared resource *upstream* of the DRAM** — the on-chip interconnect / HBM-controller path that feeds the cores — not the DRAM cells. The >200 GB/s DRAM isn't the wall; the path to it is. (To test: read-only vs write-only vs read+write ops.)

In [ ]:
fig, (a1,a2) = plt.subplots(1,2, figsize=(11,4.2))
c = np.array([x for x,_ in cores_sweep]); ct = np.array([t for _,t in cores_sweep])
a1.plot(c, ct, 'o-', color='C0'); a1.axhline(ct[1:].mean(), ls='--', color='k', alpha=.5,
        label=f'>=2 cores: ~{ct[1:].mean():.0f} us (BW-saturated)')
a1.set_xscale('log', base=2); a1.set_xticks(c); a1.set_xticklabels(c)
a1.set_xlabel('cores (SENCORES)'); a1.set_ylabel('device latency (us)')
a1.set_title('HBM BW is shared:\nflat for >=2 cores'); a1.legend(); a1.grid(alpha=.3)
d = np.array([x for x,_,_ in lx_chain]); dt = np.array([t for _,_,t in lx_chain])
a2.plot(d, dt, 'o-', color='C2', label='measured')
a2.plot(d, [predict_us(512*1024,2,lx_passes=p) for _,p,_ in lx_chain], 's--', color='C2',
        alpha=.6, label='model')
a2.set_xscale('log', base=2); a2.set_xticks(d); a2.set_xticklabels(d)
a2.set_xlabel('chain depth (# gelus, all-LX)'); a2.set_ylabel('device latency (us)')
a2.set_title('LX is ~free:\n16 chained ops ~ 1'); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## (4) The model is accurate for pointwise ops
Predicted vs measured. The four single-input activations land within ~3%. The two-input
`mul`/`add` run ~15-20% over the byte-count — an **open anomaly** (the "3-stream lower BW"
idea was **falsified**; see (5)), not a model knob.


In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
labels = [a[0] for a in accuracy]
meas = [a[3] for a in accuracy]
pred = [predict_us(a[1], a[2]) for a in accuracy]
x = np.arange(len(labels)); w = 0.38
ax.bar(x-w/2, meas, w, label='measured', color='C0')
ax.bar(x+w/2, pred, w, label='model',    color='C1')
for i,(m,p) in enumerate(zip(meas,pred)):
    ax.text(i, max(m,p)+1.3, f'{100*(p-m)/m:+.0f}%', ha='center', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels([f'{a[0]}\n({a[2]}-stream)' for a in accuracy])
ax.set_ylabel('device latency (us)'); ax.set_ylim(0, max(max(meas),max(pred))*1.22)
ax.set_title('Model vs measured (1-input ~3%; plain mul/add ~16% off -- open anomaly)')
ax.legend(); ax.grid(alpha=.3, axis='y'); plt.tight_layout(); plt.show()

## (5) Why ~111? Read+write runs at ~half either direction

The `BW_HBM ≈ 111` ceiling is not the DRAM limit — it is the cost of doing **reads and
writes together**. Pure-direction kernels at large size (effective GB/s asymptote):

- **read-only** (`sum`) ≈ **176** (86% of the 204.8 LPDDR5 peak)
- **write-only** (`b[1,N]+c[N,1]`) ≈ **146** (71%)
- **balanced read+write** (`neg`, `x+1`) ≈ **97** (47%) — *below either alone*

Every elementwise op reads inputs and writes an output, so it pays the balanced rate. *Why*
mixing ~halves throughput (read/write turnaround? half-duplex? a shared bus that saturates at
~4 cores?) is still **open** — `aiu-smi` bus-utilization is the deciding measurement. (An
earlier "stream count lowers BW" guess was **falsified**: 4- and 5-input fused adds match the
1-input rate.)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5,4))
modes = list(bw_modes); vals = [bw_modes[m] for m in modes]
bars = ax.bar(modes, vals, color=['#2a9d8f','#e9c46a','#e76f51'])
ax.axhline(PEAK_GBPS, ls='--', c='gray')
ax.text(len(modes)-0.5, PEAK_GBPS+3, f'LPDDR5 peak {PEAK_GBPS:.0f}', ha='right', c='gray')
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2, v+3, f'{v:.0f}\n{v/PEAK_GBPS*100:.0f}%', ha='center')
ax.set_ylabel('effective BW (GB/s)'); ax.set_ylim(0,220)
ax.set_title('Reads ~saturate the bus; adding writes ~halves it')
plt.tight_layout(); plt.show()


## (6) Broadcast inputs are cached — free I/O

Does a `[1,N]` operand cost a full HBM pass (re-fetched per row) or get cached (loaded once)?
Compare the full binary against the broadcast one at the same size (512×1024, device min µs):

- `add` (a + b) **58** → `bcast` (a + b[1,N]) **36**
- `mul` (a * b) **58** → `mulbcast` (a * b[1,N]) **35**

Broadcasting the second operand removes ~a full memory pass: `bcast`/`mulbcast` land on the
**2-pass** (1 read + 1 write) prediction — the broadcast operand is **cached → ~free**, and it
is not add-specific. (Scalars too: `x+1.0` ≈ `gelu`.) The model therefore **excludes broadcast
and scalar inputs** from the byte count.


In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
labels = list(bcast_pairs); vals = [bcast_pairs[k] for k in labels]
bars = ax.bar(labels, vals, color=['#e76f51','#2a9d8f','#e76f51','#2a9d8f'])
ax.axhline(TWO_PASS_US, ls='--', c='#264653')
ax.text(len(labels)-0.5, TWO_PASS_US+1.5,
        f'2-pass model {TWO_PASS_US:.0f}us (broadcast cached)', ha='right', c='#264653')
for b,v in zip(bars,vals):
    ax.text(b.get_x()+b.get_width()/2, v+1, f'{v:.0f}', ha='center')
ax.set_ylabel('device latency (us)'); ax.set_ylim(0,68)
ax.set_title('Broadcast 2nd operand -> drops a full pass (cached / free)')
plt.tight_layout(); plt.show()


## Why it matters: the model explains LX planning
A direct payoff. softmax[512×1024]: turning **LX scratchpad planning on** keeps an
intermediate on-chip, removing one HBM round-trip — **22% faster**. The model captures this
exactly: moving a tensor HBM→LX subtracts its HBM bytes (and LX is ~free).

In [ ]:
fig, ax = plt.subplots(figsize=(5,4))
ks = list(softmax_lx); vs = [softmax_lx[k] for k in ks]
ax.bar(ks, vs, color=['C2','C3'])
for i,v in enumerate(vs): ax.text(i, v+1, f'{v:.1f} us', ha='center')
d = (softmax_lx['LX off']-softmax_lx['LX on'])/softmax_lx['LX off']*100
ax.set_ylabel('device latency (us)'); ax.set_ylim(0, max(vs)*1.2)
ax.set_title(f'LX planning removes an HBM round-trip\nsoftmax[512x1024]: {d:.0f}% faster')
plt.tight_layout(); plt.show()

## What we established (pointwise)
- Latency = **fixed (~20 µs, op-independent) + HBM bytes / BW**; pointwise is bandwidth-bound.
- **Arithmetic is free** (activation-independent).
- **HBM BW is shared** — flat for ≥2 cores (core count is not a direct term).
- **Broadcast & scalar inputs are cached → ~free** (excluded from the byte count) — (6).
- **LX is ~free** (~29× HBM) — placement removes HBM passes (the source of LX speedups).
- **`BW≈111` is the balanced read+write rate**; read-only ~176, write-only ~146 (mixing ~halves it) — (5).
- Predicts single-input pointwise to **~3%**; plain 2-input `mul`/`add` ~15-20% over (open
  anomaly; NOT a stream-count BW law, which was falsified).

## Next steps
1. **`aiu-smi`** DDR-bandwidth + bus-utilization → pin the *mechanism* of the read+write penalty.
2. **Reductions** — initial model built (read full input @ ~176 + cross-core ring combine); calibrating.
3. **`mul`/`add` ~20% anomaly**, then **matmul** (compute-bound) and **tile (coarse) fusion**.
